# Stochastic Beam Alignment in Sub-THz Systems

This notebook evaluates different beam-search strategies for initial beam
alignment in sub-THz communication systems using Monte Carlo simulation.

The analysis considers both 2D and 3D scenarios.

### Assumptions

- The receiver (Rx) remains perfectly aligned toward the transmitter (Tx).
- The initial Tx steering direction is random.
- Directional antennas are modeled using ideal sector/cone patterns.
- Each beam probing attempt takes 1 ms.
- Alignment occurs when the receiver lies within the transmitter beamwidth.

### Beam-Search Strategies

1. Clockwise sequential scanning
2. Counter clockwise sequential scanning
3. Random probing with repetition
4. Random probing without repetition
5. Variable step structured scanning
6. Hybrid grid based random scanning

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import numpy as np
# Fix the random seed so simulation results are reproducible
np.random.seed(42)

Geometry Helpers

In [ ]:
def angle_diff(a, b):
    """Return the smallest angular difference between two angles."""
    d = abs(a - b)
    return min(d, 360 - d)

def is_aligned(tx_angle, target_angle, alpha):
    """Check whether the target lies within the transmitter beam."""
    return angle_diff(tx_angle, target_angle) <= alpha / 2

Strategy 1: Clockwise scanning

In [ ]:
TARGET_ANGLE = 0.0  # RX direction (reference)

def strategy_clockwise(alpha):
    """
    Strategy 1: Clockwise sequential scanning.

    The transmitter starts from a random orientation and rotates
    clockwise by 1 degree after each unsuccessful probing attempt.

    Each probing attempt corresponds to 1 ms.
    """
    theta0 = np.random.uniform(0, 360)
    time_ms = 0

    for k in range(360):
        theta = (theta0 + k) % 360
        time_ms += 1

        if is_aligned(theta, TARGET_ANGLE, alpha):
            return time_ms

    return time_ms

Strategy 2: Counter-clockwise scanning

In [ ]:
def strategy_counter_clockwise(alpha):
    """
    Strategy 2: TX rotates counter-clockwise
    """
    theta0 = np.random.uniform(0, 360)
    time_ms = 0

    for k in range(360):
        theta = (theta0 - k) % 360
        time_ms += 1

        if is_aligned(theta, TARGET_ANGLE, alpha):
            return time_ms

    return time_ms

Strategy 3: Random search

In [ ]:
def strategy_random(alpha, max_steps=10000):
    """
    Strategy 3: Random beam search with repetition.

    A new steering direction is sampled uniformly at random for every
    probing attempt. Previously tested directions may therefore be
    selected again.
    """
    time_ms = 0

    for _ in range(max_steps):
        theta = np.random.uniform(0, 360)
        time_ms += 1

        if is_aligned(theta, TARGET_ANGLE, alpha):
            return time_ms

    return time_ms

Strategy 4: Random Search Without Repetition

In [ ]:
def strategy_random_unique(alpha):
    """
    Strategy 4: Random beam search without repetition.

    All integer steering angles are generated once and shuffled.
    Each direction is therefore tested at most once.
    """
    # Create all possible integer angles
    angles = np.arange(0, 360)

    # Shuffle them randomly
    np.random.shuffle(angles)

    time_ms = 0

    for theta in angles:
        time_ms += 1
        if is_aligned(theta, TARGET_ANGLE, alpha):
            return time_ms

    return time_ms


Strategy 5: Variable-Step Sequential Scanning

In [ ]:
def strategy_variable_step(alpha, step_factor=0.5):
    """
    Strategy 5: Sequential scanning with variable step size
    
    Instead of scanning in fixed 1-degree increments, the angular step
    is calculated as a fraction of the beamwidth.
    step_factor ∈ [0.5, 1.0]  → step = step_factor * alpha
    """
    step = max(1, int(step_factor * alpha))  # ensure at least 1°

    theta0 = np.random.uniform(0, 360)
    time_ms = 0

    theta = theta0
    scanned = 0

    while scanned < 360:
        time_ms += 1
        if is_aligned(theta, TARGET_ANGLE, alpha):
            return time_ms

        theta = (theta + step) % 360
        scanned += step

    return time_ms


Strategy 6: Hybrid Grid-Based Random Scanning

In [ ]:
def strategy_random_variable_step(alpha, step_factor=0.5):
    """
    Strategy 6: Randomized variable step-size beam search

    Candidate directions are generated using a beamwidth-dependent
    angular step and then shuffled to create a randomized probing order
    without repetition.
    """
    step = max(1, int(step_factor * alpha))

    # Generate discrete angles using variable step
    theta0 = np.random.uniform(0, 360)
    angles = (theta0 + np.arange(0,360,step)) % 360
    

    # Randomize order (no repetition)
    np.random.shuffle(angles)

    time_ms = 0

    for theta in angles:
        time_ms += 1
        if is_aligned(theta, TARGET_ANGLE, alpha):
            return time_ms

    return time_ms


Monte Carlo simulation

In [ ]:
def run_simulation_full(alpha, n_trials=10000):
    """
    Run Monte Carlo simulations for all six 2D beam-search strategies.

    Parameters
    ----------
    alpha : float
        Beamwidth in degrees.
    n_trials : int
        Number of Monte Carlo trials.

    Returns
    -------
    tuple of np.ndarray
        Alignment times for strategies S1-S6.
    """
    t1, t2, t3, t4, t5, t6 = [], [], [], [], [], []

    for _ in range(n_trials):
        t1.append(strategy_clockwise(alpha))
        t2.append(strategy_counter_clockwise(alpha))
        t3.append(strategy_random(alpha))
        t4.append(strategy_random_unique(alpha))
        t5.append(strategy_variable_step(alpha, step_factor=0.5))
        t6.append(strategy_random_variable_step(alpha, step_factor=0.5))

    return (
        np.array(t1),
        np.array(t2),
        np.array(t3),
        np.array(t4),
        np.array(t5),
        np.array(t6),
    )


Mean time vs beamwidth plot

In [ ]:
alphas = [2, 5, 10, 20, 30, 60, 90]

means = []

for alpha in alphas:
    t1, t2, t3, t4, t5, t6 = run_simulation_full(alpha, 100000)
    means.append((
        t1.mean(),  # Clockwise
        t2.mean(),  # Counter-clockwise
        t3.mean(),  # Random
        t4.mean(),  # Random (no repetition)
        t5.mean(),  # Variable step scanning
        t6.mean(),  # Hybrid (Strategy 6)
    ))

m1, m2, m3, m4, m5, m6 = zip(*means)

plt.figure(figsize=(9, 5))
plt.plot(alphas, m1, '-o', label='Clockwise')
plt.plot(alphas, m2, '-s', label='Counter-clockwise')
plt.plot(alphas, m3, '-^', label='Random')
plt.plot(alphas, m4, '-d', label='Random (No repetition)')
plt.plot(alphas, m5, '-x', label='Variable step scanning')
plt.plot(alphas, m6, '-*', label='Hybrid (Random [No repetition] + Variable step scanning)')

plt.xlabel("Beamwidth α (deg)")
plt.ylabel("Mean alignment time (ms)")
plt.legend(loc='upper center',
    bbox_to_anchor=(0.5, -0.18),
    ncol=2,          # number of column
    fontsize=10)
plt.tight_layout()

plt.grid(True)
plt.show()


In [ ]:
table = []

for i, alpha in enumerate(alphas):
    table.append([
        alpha,
        round(m1[i], 2),
        round(m2[i], 2),
        round(m3[i], 2),
        round(m4[i], 2),
        round(m5[i], 2),
        round(m6[i], 2),
    ])

    headers = [
    "α (deg)",
    "Clockwise",
    "Counter-CW",
    "Random",
    "Random (no rep)",
    "Variable step",
    "Hybrid",
]
print("-" * 100)
print("{:<8} {:<12} {:<12} {:<10} {:<16} {:<14} {:<12}".format(*headers))
print("-" * 100)

for row in table:
    print("{:<8} {:<12} {:<12} {:<10} {:<16} {:<14} {:<12}".format(*row))

print("-" * 100)

PDF

In [ ]:
def plot_pdf(data, label):
    max_t = int(np.max(data))
    bins = np.arange(0.5, max_t + 1.5, 1)  # bins centered at integers

    plt.hist(
        data,
        bins=bins,
        density=True,
        alpha=0.6,
        label=label
    )


plt.figure(figsize=(8, 5))
plot_pdf(t1, "Clockwise")
plot_pdf(t2, "Counter-CW")
plot_pdf(t3, "Random")
plot_pdf(t4, "Random (no rep)")
plot_pdf(t5, "Variable step")
plot_pdf(t6, "Hybrid")

plt.xlabel("Alignment time (ms)")
plt.ylabel("Probability density")
plt.title("PDF of Alignment Time (2D, α = 10°)")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()


# 3D Beam Alignment

The beam-alignment strategies are extended from a 2D angular search
to a 3D spherical search space.

Beam directions are represented as unit vectors. Alignment occurs when
the angular separation between the transmitter beam direction and the
receiver direction is smaller than half of the beamwidth.

In [ ]:

def random_unit_vector():
    """
    Uniform random direction on the sphere.
    """
    # Sample z uniformly to avoid clustering directions near the poles
    z = np.random.uniform(-1.0, 1.0)
    # Random azimuth angle
    phi = np.random.uniform(0.0, 2*np.pi)
    # Radius of the corresponding circle at height z
    r = np.sqrt(1 - z*z)
    return np.array([r*np.cos(phi), r*np.sin(phi), z])

def angle_between(u, v):
    """
    Angle between two unit vectors in degrees.
    """
    dot = np.clip(np.dot(u, v), -1.0, 1.0)
    return np.degrees(np.arccos(dot))

def is_aligned_3d(u_tx, u_target, alpha):
    """Check whether the target direction lies inside the Tx beam."""
    return angle_between(u_tx, u_target) <= alpha / 2

def unit_from_az_el(az_deg, el_deg):
    """Convert azimuth and elevation angles to a 3D unit vector."""
    az = np.radians(az_deg)
    el = np.radians(el_deg)
    return np.array([
        np.cos(el) * np.cos(az),
        np.cos(el) * np.sin(az),
        np.sin(el)
    ])

U_TARGET = np.array([0.0, 0.0, 1.0])  # direction from TX to RX

In [ ]:
def raster_scan_3d(alpha, daz=1.0, del_=1.0, az_direction=+1, max_steps=200000):
    """
    Perform a deterministic raster scan over the 3D search space.

    The search progresses through elevation levels while sweeping
    through azimuth angles at each level.

    az_direction = +1 : clockwise azimuth sweep
    az_direction = -1 : counter-clockwise azimuth sweep
    """
    time_ms = 0

    # Random initial orientation
    az0 = np.random.uniform(0.0, 360.0)
    el0 = np.random.uniform(-90.0, 90.0)

    # Elevation grid
    el_levels = np.arange(-90.0, 90.0 + 1e-9, del_)
    i0 = int(np.argmin(np.abs(el_levels - el0)))
    el_levels = np.concatenate([el_levels[i0:], el_levels[:i0]])

    # Azimuth grid
    az_levels = np.arange(0.0, 360.0, daz)
    j0 = int(np.argmin(np.abs(az_levels - az0)))
    az_levels = np.concatenate([az_levels[j0:], az_levels[:j0]])

    # Apply direction
    if az_direction < 0:
        az_levels = az_levels[::-1]
    # Raster scan
    for el in el_levels:
        for az in az_levels:
            time_ms += 1
            u = unit_from_az_el(az, el)

            if is_aligned_3d(u, U_TARGET, alpha):
                return time_ms

            if time_ms >= max_steps:
                return time_ms

    return time_ms


In [ ]:
def strategy_clockwise_3d(alpha, max_steps=200000):
    """
    Strategy 1 (3D): Clockwise sequential scanning.

    Performs a deterministic raster scan with a clockwise
    azimuth sweep.
    """
    return raster_scan_3d(
        alpha=alpha,
        daz=1.0,
        del_=1.0,
        az_direction=+1,
        max_steps=max_steps
    )

In [ ]:
def strategy_counter_clockwise_3d(alpha, max_steps=200000):
    """
    Strategy 2 (3D):
    Deterministic raster scan with counter-clockwise azimuth sweep.
    """
    return raster_scan_3d(
        alpha=alpha,
        daz=1.0,
        del_=1.0,
        az_direction=-1,
        max_steps=max_steps
    )

In [ ]:

def strategy_random_3d(alpha, max_steps=200000):
    """
    Strategy 3 (3D): Random probing with repetition.

    A new direction is sampled uniformly from the sphere for every
    probing attempt. Previously tested directions may be sampled again.
    """
    time_ms = 0
    for _ in range(max_steps):
        u = random_unit_vector()
        time_ms += 1
        if is_aligned_3d(u, U_TARGET, alpha):
            return time_ms
    return time_ms


In [ ]:
def strategy_random_no_rep_3d_FIXED(alpha, step_factor=0.7, max_steps=200000):
    """
    Strategy 4 (3D): Random probing without repetition.

    A geometry-aware spherical grid is generated first. Candidate
    directions are then shuffled so that each grid direction is
    probed at most once.
    """
    step = max(1, int(step_factor * alpha))
    time_ms = 0

    # Equal-area elevation sampling
    el_vals = np.linspace(-1.0, 1.0, int(180 / step) + 1)
    el_levels = np.degrees(np.arcsin(el_vals))

    grid = []

    for el in el_levels:
        circumference_factor = np.cos(np.radians(el))

        if circumference_factor < 0.1:
            az_levels = [0.0]
        else:
            adj_step = step / circumference_factor
            az_levels = np.arange(0.0, 360.0, adj_step)

        az0 = np.random.uniform(0.0, 360.0)
        for az in az_levels:
            az_cur = (az + az0) % 360.0
            grid.append((az_cur, el))

    np.random.shuffle(grid)

    for az, el in grid:
        time_ms += 1
        u = unit_from_az_el(az, el)

        if is_aligned_3d(u, U_TARGET, alpha):
            return time_ms

        if time_ms >= max_steps:
            return time_ms

    return time_ms


In [ ]:
def strategy_variable_step_scan_3d_FIXED(alpha, step_factor=0.7):
    """
    Strategy 5 (3D): Structured variable-step scanning.

    The search grid adapts its azimuth resolution according to
    elevation to account for the geometry of the sphere.
    """
    step = max(1, int(step_factor * alpha))
    time_ms = 0
    
    # Generate systematic elevation levels
    el_levels = np.arange(-90.0, 90.0, step)
    
    # Randomize the starting elevation while preserving scan order
    start_idx = np.random.randint(0, len(el_levels))
    el_levels = np.roll(el_levels, -start_idx)

    for el in el_levels:
        # Reduce the number of azimuth samples toward the poles
        circumference_factor = np.cos(np.radians(el))
        
        if circumference_factor < 0.1: # Near poles
            current_az_levels = [0.0] 
        else:
            # Increase the azimuth step size toward the poles
            adj_step = step / circumference_factor
            current_az_levels = np.arange(0.0, 360.0, adj_step)
            
        # Randomize start azimuth for this specific ring
        az0 = np.random.uniform(0, 360)
        
        for az in current_az_levels:
            time_ms += 1
            az_cur = (az + az0) % 360.0
            u = unit_from_az_el(az_cur, el)

            if is_aligned_3d(u, U_TARGET, alpha):
                return time_ms
    return time_ms

In [ ]:
def strategy_hybrid_3d_FIXED(alpha, step_factor=0.7, max_steps=200000):
    """
    Strategy 6 (3D): Hybrid grid-based random scanning.

    A geometry-aware spherical grid is generated using elevation-
    dependent azimuth spacing. The complete grid is then shuffled
    to create a randomized probing order.
    """
    step = max(1, int(step_factor * alpha))
    time_ms = 0

    # Equal-area elevation sampling
    el_vals = np.linspace(-1.0, 1.0, int(180 / step) + 1)
    el_levels = np.degrees(np.arcsin(el_vals))

    grid = []

    for el in el_levels:
        circumference_factor = np.cos(np.radians(el))

        if circumference_factor < 0.1:
            az_levels = [0.0]  # Pole handling
        else:
            adj_step = step / circumference_factor
            az_levels = np.arange(0.0, 360.0, adj_step)

        # Random azimuth offset per ring
        az0 = np.random.uniform(0.0, 360.0)
        for az in az_levels:
            az_cur = (az + az0) % 360.0
            grid.append((az_cur, el))

    # Randomize probing order (hybrid aspect)
    np.random.shuffle(grid)

    for az, el in grid:
        time_ms += 1
        u = unit_from_az_el(az, el)

        if is_aligned_3d(u, U_TARGET, alpha):
            return time_ms

        if time_ms >= max_steps:
            return time_ms

    return time_ms


In [ ]:
def run_simulation_full_3d(alpha, n_trials=50000):
    """
    Run Monte Carlo simulations for all six 3D beam-search strategies.
    """
    T1, T2, T3, T4, T5, T6 = [], [], [], [], [], []

    for _ in range(n_trials):
        T1.append(strategy_clockwise_3d(alpha))
        T2.append(strategy_counter_clockwise_3d(alpha))
        T3.append(strategy_random_3d(alpha))
        T4.append(strategy_random_no_rep_3d_FIXED(alpha))
        T5.append(strategy_variable_step_scan_3d_FIXED(alpha))
        T6.append(strategy_hybrid_3d_FIXED(alpha))

    return (
        np.array(T1),
        np.array(T2),
        np.array(T3),
        np.array(T4),
        np.array(T5),
        np.array(T6),
    )


In [ ]:

alphas = [5, 10, 20, 30, 60, 90]

means = []

for a in alphas:
        t1, t2, t3, t4, t5, t6= run_simulation_full_3d(a, n_trials=10000)
        means.append((
        t1.mean(),
        t2.mean(),
        t3.mean(),
        t4.mean(),
        t5.mean(),
        t6.mean(),
    ))

m1, m2, m3, m4, m5, m6 = zip(*means)

plt.figure(figsize=(9, 6))

plt.plot(alphas, m1, '-o', label='Clockwise (3D)')
plt.plot(alphas, m2, '-s', label='Counter-clockwise (3D)')
plt.plot(alphas, m3, '-^', label='Random (3D)')
plt.plot(alphas, m4, '-d', label='Random (no rep, 3D)')
plt.plot(alphas, m5, '-x', label='Variable step (3D)')
plt.plot(alphas, m6, '-*', label='Hybrid (3D)')

plt.xlabel("Beamwidth α (deg)")
plt.ylabel("Mean alignment time (ms)")
plt.title("Mean Alignment Time vs Beamwidth (3D)")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()



In [ ]:
def plot_pdf(data, label, bins=50):
    plt.hist(
        data,
        bins=bins,
        density=True,
        alpha=0.6,
        label=label
    )

plt.figure(figsize=(8, 5))
plot_pdf(t1, "Clockwise (3D)")
plot_pdf(t2, "Counter-CW (3D)")
plot_pdf(t3, "Random (3D)")
plot_pdf(t4, "Random (no rep, 3D)")
plot_pdf(t5, "Variable step (3D)")
plot_pdf(t6, "Hybrid (3D)")

plt.xlabel("Alignment time (ms)")
plt.ylabel("Probability density")
plt.title("PDF of Alignment Time (3D, α = 90°)")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()